# 07 Leak check, step 1: cost timing and the UC3M plate-text chance table (M1b-7)
Runs on Colab (Drive mounted). Plan: PLAN.md (M1b-7) and the decisions of 2026-11-14.

**This notebook only measures.** It times ORB for checks 1 (US test vs CCPD train), 3 (UC3M test vs UC3M train) and 4 (CCPD held-out vs CCPD train), and compares the plate text of the authors' UC3M test plates with their train plates. It removes nothing, sets no threshold and starts no long run. Random pairs here are used for **cost only**, never for a threshold.

**Output rules:** no images and no plate text are printed (counts and timings only), so no cell needs the `remove_output` tag. Restart & Run All before committing.

In [ ]:
import csv, hashlib, io, os, pickle, random, shutil, tarfile, time
from collections import Counter
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
from google.colab import drive

In [ ]:
# Parameters
SEED = 0
ORB_FEATURES, ORB_RATIO, ORB_MIN_GOOD = 500, 0.75, 8   # same as notebooks 03 and 04
N_IMG = 50          # images timed for the UC3M decode/describe steps (32 MP files)
N_IMG_SMALL = 200   # images timed for the CCPD / US describe steps
N_PAIRS = 200       # random pairs timed per combination (cost only)
N_COPY = 20         # image-vs-altered-copy pairs timed per combination (these reach RANSAC)
UC3M_SIZES = (1024, 1536)   # candidate long sides for UC3M; the choice is made later from recall on a positive control
DRIVE = '/content/drive/MyDrive/alpr'
ARCHIVES = f'{DRIVE}/archives'
LOCAL = '/content/tars'
TARS = {   # name -> (file, MD5 recorded in PLAN.md)
    'ccpd': ('ccpd_yolo.tar', 'bd44986acd6691a4f90fbea1e6017682'),
    'us': ('us_yolo.tar', '1ce491ae6a7accefdb0b9156a30f2acc'),
    'uc3m': ('uc3m_yolo.tar', 'e3ef306731f33095bac983338c6adb04'),
}
LABELS_CSV = f'{ARCHIVES}/uc3m_labels.csv'   # contains partial plate text: Drive only, never printed

In [ ]:
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
cd /content
if [ -d vehicle-plate-recognition ]; then git -C vehicle-plate-recognition pull --ff-only; else git clone https://github.com/medrees-1000/vehicle-plate-recognition.git; fi
pip install -q uv  # approved raw pip: only to bootstrap uv on Colab
uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition

In [ ]:
# setup: copy the three tars to the VM disk, check MD5, list members (names stay in memory, nothing is printed)
from alpr import ccpd, uc3m   # after the install cell (ImportError on the first run? restart the kernel once)
Path(LOCAL).mkdir(exist_ok=True)
MEM = {}   # name -> {split: [member names]}
for name, (fn, md5) in TARS.items():
    src, dst = Path(ARCHIVES, fn), Path(LOCAL, fn)
    assert src.exists(), f'{fn} not found in {ARCHIVES}'
    if not dst.exists():
        t0 = time.time(); shutil.copyfile(src, dst); print(f'{name}: copied in {time.time() - t0:.0f} s')
    got = ccpd.file_md5(dst)
    assert got == md5, f'{name}: MD5 {got} differs from PLAN.md ({md5})'
    with tarfile.open(dst) as t:
        MEM[name] = {}
        for m in t.getmembers():
            if m.isfile() and m.name.startswith('images/'):
                MEM[name].setdefault(m.name.split('/')[1], []).append(m.name)
    print(name, 'MD5 ok;', {s: len(v) for s, v in MEM[name].items()}, 'images')
assert len(MEM['ccpd']['train']) == 16556 and len(MEM['ccpd']['heldout']) == 300 and len(MEM['us']['test']) == 105
assert len(MEM['uc3m']['train']) == 1580 and len(MEM['uc3m']['test']) == 395

In [ ]:
# helpers (ORB as in notebooks 03/04; timed pieces are split so the RANSAC share is visible)
TAR = {n: tarfile.open(Path(LOCAL, fn)) for n, (fn, _) in TARS.items()}
def read(name, member): return TAR[name].extractfile(member).read()

def load_gray(data, long_side=None):
    im = Image.open(io.BytesIO(data))
    if long_side:
        im.draft('L', (long_side, long_side))   # JPEG DCT scaling: decodes at 1/2, 1/4 or 1/8 size when that is still >= the request
    g = np.asarray(im.convert('L'))
    if long_side and max(g.shape) > long_side:
        s = long_side / max(g.shape)
        g = cv2.resize(g, (round(g.shape[1] * s), round(g.shape[0] * s)), interpolation=cv2.INTER_AREA)
    return g

ORB = cv2.ORB_create(nfeatures=ORB_FEATURES)
def describe(g):
    kp, des = ORB.detectAndCompute(g, None)
    return (np.float32([k.pt for k in kp]), des) if des is not None and len(kp) else (np.zeros((0, 2), np.float32), None)

BF = cv2.BFMatcher(cv2.NORM_HAMMING)
def score_timed(a, b):
    """(inliers, seconds of knn+ratio, seconds of RANSAC or None when fewer than ORB_MIN_GOOD good matches)."""
    (pa, da), (pb, db) = a, b
    if da is None or db is None or len(da) < 2 or len(db) < 2:
        return 0, 0.0, None
    t0 = time.perf_counter()
    good = [m[0] for m in BF.knnMatch(da, db, k=2) if len(m) == 2 and m[0].distance < ORB_RATIO * m[1].distance]
    t1 = time.perf_counter()
    if len(good) < ORB_MIN_GOOD:
        return 0, t1 - t0, None
    M, inl = cv2.estimateAffinePartial2D(pa[[m.queryIdx for m in good]], pb[[m.trainIdx for m in good]], method=cv2.RANSAC, ransacReprojThreshold=4.0)
    return (int(inl.sum()) if M is not None else 0), t1 - t0, time.perf_counter() - t1

def altered(g, rng):   # synthetic altered copy: rotate up to 10 deg, crop up to 15%, noise
    h, w = g.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-10, 10), 1.0)
    r = cv2.warpAffine(g, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    c = rng.uniform(0, 0.15); y, x = int(h * c / 2), int(w * c / 2)
    r = r[y:h - y, x:w - x].astype(np.float32) + np.random.RandomState(rng.randrange(2**31)).normal(0, 6, (h - 2 * y, w - 2 * x))
    return np.clip(r, 0, 255).astype(np.uint8)

def ms(x): return f'{x * 1000:.2f} ms'

In [ ]:
# 1. image cost: read, decode(+resize), describe (50 UC3M images, 200 of the others; cost only)
rng = random.Random(SEED)
SAMPLE = {}   # (name, split, size) -> (list of gray arrays, descriptors)
def time_images(name, split, n, size):
    members = rng.sample(MEM[name][split], n)
    t0 = time.perf_counter(); raw = [read(name, m) for m in members]; t_read = (time.perf_counter() - t0) / n
    t0 = time.perf_counter(); grays = [load_gray(d, size) for d in raw]; t_dec = (time.perf_counter() - t0) / n
    t0 = time.perf_counter(); des = [describe(g) for g in grays]; t_des = (time.perf_counter() - t0) / n
    SAMPLE[(name, split, size)] = (grays, des)
    kp = np.mean([len(d[0]) for d in des])
    print(f'{name}/{split} size={size}: n={n} read {ms(t_read)} | decode+resize {ms(t_dec)} | describe {ms(t_des)} | mean keypoints {kp:.0f} | mean image {np.mean([g.shape[1] for g in grays]):.0f}x{np.mean([g.shape[0] for g in grays]):.0f}')
    return t_read + t_dec + t_des
PER_IMG = {}
for size in UC3M_SIZES:
    for split in ('train', 'test'):
        PER_IMG[('uc3m', split, size)] = time_images('uc3m', split, N_IMG, size)
for name, split in (('ccpd', 'train'), ('ccpd', 'heldout'), ('us', 'test')):
    PER_IMG[(name, split, None)] = time_images(name, split, min(N_IMG_SMALL, len(MEM[name][split])), None)

In [ ]:
# 2. pair cost: random pairs (cost only) and image-vs-altered-copy pairs (these reach RANSAC)
def time_pairs(label, qa, ta):
    (_, qd), (_, td) = SAMPLE[qa], SAMPLE[ta]
    qg = SAMPLE[qa][0]
    r = random.Random(SEED + 1)
    res = [score_timed(qd[r.randrange(len(qd))], td[r.randrange(len(td))]) for _ in range(N_PAIRS)]
    knn = np.mean([x[1] for x in res]); ran = [x[2] for x in res if x[2] is not None]
    cp = []
    for i in range(min(N_COPY, len(qg))):
        cp.append(score_timed(qd[i], describe(altered(qg[i], r))))
    cp_ran = [x[2] for x in cp if x[2] is not None]
    ransac = np.mean(ran + cp_ran) if ran + cp_ran else None
    print(f'{label}: knn+ratio {ms(knn)}/pair | random pairs that reached RANSAC: {len(ran)}/{N_PAIRS} | altered copies that reached RANSAC: {len(cp_ran)}/{len(cp)} (inliers {sorted(x[0] for x in cp)[:3]}..) | RANSAC {ms(ransac) if ransac is not None else "not measured"}/call')
    return knn, (len(ran) / N_PAIRS), ransac
PAIR = {}
PAIR['1'] = time_pairs('check 1  US test x CCPD train', ('us', 'test', None), ('ccpd', 'train', None))
PAIR['4'] = time_pairs('check 4  CCPD held-out x CCPD train', ('ccpd', 'heldout', None), ('ccpd', 'train', None))
for size in UC3M_SIZES:
    PAIR[f'3@{size}'] = time_pairs(f'check 3  UC3M test x train @ {size}', ('uc3m', 'test', size), ('uc3m', 'train', size))

In [ ]:
# 3. cost table: pairs x measured per-pair time (single process). This is arithmetic on the sample above, not a full-run measurement
CHECKS = {'1': (105, 16556, ('us', 'test', None), ('ccpd', 'train', None)),
          '4': (300, 16556, ('ccpd', 'heldout', None), ('ccpd', 'train', None)),
          **{f'3@{s}': (395, 1580, ('uc3m', 'test', s), ('uc3m', 'train', s)) for s in UC3M_SIZES}}
print(f'cores: os.cpu_count() = {os.cpu_count()}, cv2.getNumThreads() = {cv2.getNumThreads()} (whether knnMatch uses them was not measured)')
for k, (nq, nt, q, t) in CHECKS.items():
    knn, frac, ransac = PAIR[k]
    per = knn + frac * (ransac or 0)
    n = nq * nt
    desc = nq * PER_IMG[q] + nt * PER_IMG[t]   # describe both sides once (a cache on Drive makes this one-off)
    print(f'check {k}: {n:,} pairs x {ms(per)} = {n * per / 3600:.2f} h (random-pair share of RANSAC: {frac:.3f}; RANSAC on true copies not included) | one-off image reading + describe: {desc / 60:.1f} min')
print('Not measured: RANSAC cost on the real tail of the pairs, Drive read speed during the full run, multi-process speed-up.')

In [ ]:
# 4. UC3M plate text: authors' test plates vs train plates, counts only (no text printed)
rows = list(csv.DictReader(open(LABELS_CSV, newline='', encoding='utf-8')))
test_lp = [r['lp_id'] for r in rows if r['split'] == 'test']
train_lp = [r['lp_id'] for r in rows if r['split'] == 'train']
cnt = uc3m.plate_pair_counts(test_lp, train_lp)
print(f'plates: test {len(test_lp)}, train {len(train_lp)}; equal-length pairs compared: {sum(cnt.values()):,}')
print('k = positions where both plates are visible; match = all k agree; near = exactly one of the k differs')
print(f'{"k":>3} {"pairs":>10} {"match":>7} {"near":>7} {"match rate":>11}')
for k in sorted({k for k, _ in cnt}):
    n = sum(v for (kk, _), v in cnt.items() if kk == k)
    m, nr = cnt.get((k, k), 0), cnt.get((k, k - 1), 0)
    print(f'{k:>3} {n:>10,} {m:>7} {nr:>7} {m / n:>11.2e}')

**Stop here.** Report the timings and the table to the author; do not start any long run before the reply (PLAN.md, M1b-7).